# 20. SAM2 Human-in-the-Loop 잎 어노테이터

**목적**: 이미지를 보면서 잎을 클릭 → SAM2가 마스크 생성 → YOLO v11 seg용 polygon txt 저장

**조작법**:
- **Left-click** 이미지 위 잎 중심 → SAM2 마스크 미리보기
- **Right-click** → 해당 위치를 negative point로 추가 (마스크 제외 영역 지정)
- **Add Leaf** → 현재 마스크를 확정하고 다음 잎으로
- **Clear** → 현재 잎의 포인트 초기화
- **Save** → 확정된 모든 마스크를 YOLO txt로 저장, parquet 업데이트
- **Skip** → 이 이미지 건너뜀
- **Prev / Next** → 이미지 탐색

In [ ]:
%matplotlib widget
import os, warnings
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path
from PIL import Image

warnings.filterwarnings('ignore')

try:
    import google.colab
    IN_COLAB = True
    ROOT = Path('/content')
except ImportError:
    IN_COLAB = False
    ROOT = Path(os.getcwd()).parent

DATASET_DIR = ROOT / 'dataset' / 'collector'
LABELS_DIR  = DATASET_DIR / 'yolo_labels'
LABELS_DIR.mkdir(parents=True, exist_ok=True)

SAM2_CKPT_DIR  = ROOT / 'checkpoints'
SAM2_CKPT_DIR.mkdir(exist_ok=True)
SAM2_CKPT_PATH = SAM2_CKPT_DIR / 'sam2.1_hiera_small.pt'
SAM2_CFG       = 'configs/sam2.1/sam2.1_hiera_s.yaml'

LEAF_CLASS_ID  = 0

print(f'ROOT       : {ROOT}')
print(f'LABELS_DIR : {LABELS_DIR}')
print(f'CKPT       : {SAM2_CKPT_PATH}')

In [ ]:
# SAM2 체크포인트 다운로드 (없을 때만)
import urllib.request

CKPT_URL = (
    'https://dl.fbaipublicfiles.com/segment_anything_2/092824/'
    'sam2.1_hiera_small.pt'
)

if not SAM2_CKPT_PATH.exists():
    print('체크포인트 다운로드 중...')
    def _progress(block, block_size, total):
        pct = block * block_size / total * 100
        print(f'\r  {pct:.1f}%', end='', flush=True)
    urllib.request.urlretrieve(CKPT_URL, SAM2_CKPT_PATH, reporthook=_progress)
    print('\n완료')
else:
    print(f'체크포인트 존재: {SAM2_CKPT_PATH}')

In [ ]:
import torch
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'Device: {DEVICE}')

sam2_model = build_sam2(SAM2_CFG, SAM2_CKPT_PATH, device=DEVICE)
predictor  = SAM2ImagePredictor(sam2_model)
print('SAM2 로드 완료')

In [ ]:
# 분할별 parquet 로드 & 어노테이션 컬럼 초기화
splits = ['train', 'val', 'test']
parquet_paths = {s: DATASET_DIR / f'{s}.parquet' for s in splits}

dfs = {s: pd.read_parquet(p) for s, p in parquet_paths.items()}

for s, df in dfs.items():
    if 'yolo_label_path' not in df.columns:
        df['yolo_label_path'] = None
    if 'leaf_count' not in df.columns:
        df['leaf_count'] = pd.NA

df_all = pd.concat(dfs.values(), ignore_index=True)
print(f'전체 이미지: {len(df_all)}')
print(f'어노테이션 완료: {df_all["yolo_label_path"].notna().sum()}')

In [ ]:
# ── 헬퍼 함수 ────────────────────────────────────────────────────────────────

COLORS = plt.cm.tab10.colors  # 잎별 색상

def mask_to_yolo_polygon(mask: np.ndarray, img_h: int, img_w: int) -> str | None:
    """
    binary mask (H,W bool) → YOLO seg 한 줄
    '0 x1 y1 x2 y2 ...'
    """
    m = (mask.astype(np.uint8) * 255)
    contours, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    largest = max(contours, key=cv2.contourArea)
    if cv2.contourArea(largest) < 10:
        return None
    epsilon = 0.003 * cv2.arcLength(largest, True)
    approx  = cv2.approxPolyDP(largest, epsilon, True).reshape(-1, 2)
    if len(approx) < 3:
        return None
    pts_norm = approx / np.array([img_w, img_h], dtype=float)
    pts_norm = np.clip(pts_norm, 0.0, 1.0)
    coords   = ' '.join(f'{v:.6f}' for v in pts_norm.flatten())
    return f'{LEAF_CLASS_ID} {coords}'


def save_yolo_label(label_path: Path, yolo_lines: list[str]) -> None:
    label_path.parent.mkdir(parents=True, exist_ok=True)
    label_path.write_text('\n'.join(yolo_lines) + '\n')


def update_parquets(dfs: dict, df_all: pd.DataFrame, parquet_paths: dict) -> None:
    for s, df in dfs.items():
        idx = df_all[df_all['split'] == s].index
        df['yolo_label_path'] = df_all.loc[idx, 'yolo_label_path'].values
        df['leaf_count']      = df_all.loc[idx, 'leaf_count'].values
        df.to_parquet(parquet_paths[s], index=False)


def overlay_mask(ax, mask: np.ndarray, color, alpha=0.45) -> None:
    rgba = np.zeros((*mask.shape, 4), dtype=float)
    rgba[mask] = [*color[:3], alpha]
    ax.imshow(rgba, interpolation='nearest')

In [ ]:
class LeafAnnotator:
    """ipywidgets + matplotlib widget 기반 Human-in-the-Loop 어노테이터."""

    def __init__(self, df_all, predictor, dfs, parquet_paths):
        self.df         = df_all.reset_index(drop=True)
        self.predictor  = predictor
        self.dfs        = dfs
        self.pq_paths   = parquet_paths
        self.idx        = self._first_unannotated()

        # 현재 이미지 상태
        self._image_rgb: np.ndarray | None = None
        self._pos_pts: list[tuple] = []   # positive clicks
        self._neg_pts: list[tuple] = []   # negative clicks (right-click)
        self._current_mask: np.ndarray | None = None
        self._confirmed_masks: list[np.ndarray] = []  # 확정된 잎 마스크들

        self._build_ui()
        self._load_image()

    # ── UI 구성 ──────────────────────────────────────────────────────────────

    def _build_ui(self):
        # matplotlib figure
        self.fig, self.ax = plt.subplots(figsize=(10, 6))
        self.fig.tight_layout()
        self.fig.canvas.mpl_connect('button_press_event', self._on_click)

        # 버튼
        btn_kw = dict(layout=widgets.Layout(width='110px', height='36px'))
        self.btn_add    = widgets.Button(description='✅ Add Leaf',  button_style='success', **btn_kw)
        self.btn_clear  = widgets.Button(description='🔄 Clear',     button_style='warning', **btn_kw)
        self.btn_save   = widgets.Button(description='💾 Save',      button_style='primary', **btn_kw)
        self.btn_skip   = widgets.Button(description='⏭ Skip',       button_style='',        **btn_kw)
        self.btn_prev   = widgets.Button(description='◀ Prev',       button_style='',        **btn_kw)
        self.btn_next   = widgets.Button(description='Next ▶',       button_style='',        **btn_kw)

        self.btn_add.on_click(self._on_add)
        self.btn_clear.on_click(self._on_clear)
        self.btn_save.on_click(self._on_save)
        self.btn_skip.on_click(self._on_skip)
        self.btn_prev.on_click(lambda _: self._nav(-1))
        self.btn_next.on_click(lambda _: self._nav(+1))

        # 상태 표시
        self.lbl_status = widgets.HTML(value='')
        self.lbl_info   = widgets.HTML(value='')

        btn_row1 = widgets.HBox([self.btn_add, self.btn_clear, self.btn_save, self.btn_skip])
        btn_row2 = widgets.HBox([self.btn_prev, self.btn_next])
        self.ui  = widgets.VBox([
            btn_row1, btn_row2,
            self.lbl_status, self.lbl_info
        ])
        display(self.ui)

    # ── 이미지 로드 ──────────────────────────────────────────────────────────

    def _first_unannotated(self) -> int:
        unannotated = self.df[self.df['yolo_label_path'].isna()].index
        return int(unannotated[0]) if len(unannotated) else 0

    def _load_image(self):
        row  = self.df.iloc[self.idx]
        path = row['image_path']
        img  = Image.open(path).convert('RGB')
        self._image_rgb = np.array(img)
        self._pos_pts.clear()
        self._neg_pts.clear()
        self._current_mask = None
        self._confirmed_masks.clear()

        self.predictor.set_image(self._image_rgb)

        self._redraw()
        annotated = self.df['yolo_label_path'].notna().sum()
        self.lbl_info.value = (
            f'<b>[{self.idx+1}/{len(self.df)}]</b> '
            f'{row["image_filename"]} &nbsp;|&nbsp; '
            f'split: {row["split"]} &nbsp;|&nbsp; '
            f'어노테이션 완료: {annotated}/{len(self.df)}'
        )
        self._set_status('🖱 잎 위를 클릭하세요 (좌클릭=포함, 우클릭=제외)', 'black')

    # ── 클릭 이벤트 ──────────────────────────────────────────────────────────

    def _on_click(self, event):
        if event.inaxes is not self.ax:
            return
        if event.xdata is None or event.ydata is None:
            return
        x, y = int(round(event.xdata)), int(round(event.ydata))
        if event.button == 1:   # left click → positive
            self._pos_pts.append((x, y))
        elif event.button == 3: # right click → negative
            self._neg_pts.append((x, y))
        else:
            return
        self._run_sam2()

    def _run_sam2(self):
        if not self._pos_pts and not self._neg_pts:
            return
        all_pts    = self._pos_pts + self._neg_pts
        all_labels = [1] * len(self._pos_pts) + [0] * len(self._neg_pts)

        coords = np.array(all_pts,    dtype=float)
        labels = np.array(all_labels, dtype=int)

        masks, scores, _ = self.predictor.predict(
            point_coords=coords,
            point_labels=labels,
            multimask_output=True,
        )
        best = masks[int(np.argmax(scores))]
        self._current_mask = best
        self._redraw()
        self._set_status(
            f'마스크 생성 완료 (score={scores.max():.3f}) — '
            f'잎 추가: ✅ Add Leaf | 재시도: 🔄 Clear', 'blue'
        )

    # ── 렌더링 ────────────────────────────────────────────────────────────────

    def _redraw(self):
        self.ax.cla()
        self.ax.imshow(self._image_rgb)
        self.ax.axis('off')

        # 확정된 마스크들
        for i, mask in enumerate(self._confirmed_masks):
            color = COLORS[i % len(COLORS)]
            overlay_mask(self.ax, mask, color, alpha=0.45)

        # 현재 미리보기 마스크 (흰색 반투명)
        if self._current_mask is not None:
            overlay_mask(self.ax, self._current_mask, (1,1,1), alpha=0.35)

        # 클릭 포인트 표시
        for (px, py) in self._pos_pts:
            self.ax.plot(px, py, 'g*', markersize=14, markeredgecolor='white', markeredgewidth=1)
        for (px, py) in self._neg_pts:
            self.ax.plot(px, py, 'r*', markersize=14, markeredgecolor='white', markeredgewidth=1)

        # 범례
        if self._confirmed_masks:
            patches = [
                mpatches.Patch(color=COLORS[i % len(COLORS)], label=f'Leaf {i+1}')
                for i in range(len(self._confirmed_masks))
            ]
            self.ax.legend(handles=patches, loc='upper right', fontsize=8)

        self.fig.canvas.draw_idle()

    # ── 버튼 콜백 ────────────────────────────────────────────────────────────

    def _on_add(self, _):
        if self._current_mask is None:
            self._set_status('⚠ 먼저 이미지를 클릭해 마스크를 생성하세요.', 'red')
            return
        self._confirmed_masks.append(self._current_mask.copy())
        self._current_mask = None
        self._pos_pts.clear()
        self._neg_pts.clear()
        n = len(self._confirmed_masks)
        self._redraw()
        self._set_status(f'✅ 잎 {n}개 확정 — 다음 잎을 클릭하거나 💾 Save 하세요.', 'green')

    def _on_clear(self, _):
        self._pos_pts.clear()
        self._neg_pts.clear()
        self._current_mask = None
        self._redraw()
        self._set_status('🔄 현재 포인트 초기화 — 다시 클릭하세요.', 'orange')

    def _on_save(self, _):
        if not self._confirmed_masks:
            self._set_status('⚠ 확정된 마스크가 없습니다. Add Leaf 먼저 하세요.', 'red')
            return
        row    = self.df.iloc[self.idx]
        h, w   = self._image_rgb.shape[:2]
        stem   = Path(row['image_filename']).stem
        lbl_path = LABELS_DIR / f'{stem}.txt'

        lines = []
        for mask in self._confirmed_masks:
            line = mask_to_yolo_polygon(mask, h, w)
            if line:
                lines.append(line)

        if not lines:
            self._set_status('⚠ 유효한 폴리곤이 없습니다. 마스크를 다시 생성하세요.', 'red')
            return

        save_yolo_label(lbl_path, lines)
        self.df.at[self.idx, 'yolo_label_path'] = str(lbl_path)
        self.df.at[self.idx, 'leaf_count']      = len(lines)
        update_parquets(self.dfs, self.df, self.pq_paths)

        self._set_status(
            f'💾 저장 완료! 잎 {len(lines)}개 → {lbl_path.name}', 'green'
        )
        self._nav(+1)

    def _on_skip(self, _):
        self._set_status('⏭ 건너뜀', 'gray')
        self._nav(+1)

    def _nav(self, delta: int):
        new_idx = self.idx + delta
        if 0 <= new_idx < len(self.df):
            self.idx = new_idx
            self._load_image()
        else:
            self._set_status('📋 모든 이미지를 검토했습니다!', 'purple')

    def _set_status(self, msg: str, color: str = 'black'):
        self.lbl_status.value = f'<span style="color:{color};font-size:13px">{msg}</span>'

In [ ]:
# 어노테이터 실행
annotator = LeafAnnotator(df_all, predictor, dfs, parquet_paths)

In [ ]:
# 진행 현황 확인 (언제든 실행 가능)
df_check = pd.concat([pd.read_parquet(p) for p in parquet_paths.values()], ignore_index=True)
done  = df_check['yolo_label_path'].notna().sum()
total = len(df_check)
print(f'어노테이션 완료: {done} / {total}  ({done/total*100:.1f}%)')
if done > 0:
    print(f'평균 잎 수: {df_check["leaf_count"].dropna().mean():.2f}')
    print(f'잎 수 분포:\n{df_check["leaf_count"].value_counts().sort_index()}')

In [ ]:
# YOLO 데이터셋 yaml 생성 (학습 준비)
import yaml

YOLO_DIR = ROOT / 'dataset' / 'yolo_seg'

# split별 이미지/라벨 경로 정리
for split in ['train', 'val', 'test']:
    (YOLO_DIR / split / 'images').mkdir(parents=True, exist_ok=True)
    (YOLO_DIR / split / 'labels').mkdir(parents=True, exist_ok=True)

df_check = pd.concat([pd.read_parquet(p) for p in parquet_paths.values()], ignore_index=True)
annotated = df_check[df_check['yolo_label_path'].notna()]

import shutil
for _, row in annotated.iterrows():
    split    = row['split']
    img_src  = Path(row['image_path'])
    lbl_src  = Path(row['yolo_label_path'])
    img_dst  = YOLO_DIR / split / 'images' / img_src.name
    lbl_dst  = YOLO_DIR / split / 'labels' / lbl_src.name
    if not img_dst.exists():
        shutil.copy2(img_src, img_dst)
    if not lbl_dst.exists():
        shutil.copy2(lbl_src, lbl_dst)

yaml_cfg = {
    'path'  : str(YOLO_DIR),
    'train' : 'train/images',
    'val'   : 'val/images',
    'test'  : 'test/images',
    'nc'    : 1,
    'names' : ['leaf'],
}
yaml_path = YOLO_DIR / 'dataset.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_cfg, f, default_flow_style=False, allow_unicode=True)

print(f'YOLO 데이터셋 yaml: {yaml_path}')
print(f'복사된 이미지/라벨 쌍: {len(annotated)}')